In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import Callback

print(tf.__version__)
print("ALL MODULES IMPORTED SUCCESSFULLY")

In [ ]:
# Path to LFW dataset in Kaggle
DATA_DIR = "/kaggle/input/lfw-dataset/lfw-deepfunneled"

# Image dimensions
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32
EPOCHS = 50

# Make sure the dataset exists
print("Actual identities in dataset:", len(os.listdir(DATA_DIR)))

In [ ]:
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.1
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='input',      # input == output for autoencoder
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='input',
    subset='validation',
    shuffle=True
)

In [ ]:
class Face_Autoencoder:
    def __init__(self,
                 input_dim,
                 encoder_conv_filters,
                 encoder_conv_kernel_size,
                 encoder_conv_strides,
                 decoder_conv_t_filters,
                 decoder_conv_t_kernel_size,
                 decoder_conv_t_strides,
                 z_dim,
                 use_batch_norm=False,
                 use_dropout=False):
        
        self.input_dim = input_dim
        self.encoder_conv_filters = encoder_conv_filters
        self.encoder_conv_kernel_size = encoder_conv_kernel_size
        self.encoder_conv_strides = encoder_conv_strides

        self.decoder_conv_t_filters = decoder_conv_t_filters
        self.decoder_conv_t_kernel_size = decoder_conv_t_kernel_size
        self.decoder_conv_t_strides = decoder_conv_t_strides

        self.z_dim = z_dim
        self.use_batch_norm = use_batch_norm
        self.use_dropout = use_dropout

        self.n_layers_encoder = len(encoder_conv_filters)
        self.n_layers_decoder = len(decoder_conv_t_filters)

    def build(self):
        # Encoder
        encoder_input = layers.Input(shape=self.input_dim, name='encoder_input')
        x = encoder_input

        for i in range(self.n_layers_encoder):
            x = layers.Conv2D(
                filters=self.encoder_conv_filters[i],
                kernel_size=self.encoder_conv_kernel_size[i],
                strides=self.encoder_conv_strides[i],
                padding='same',
                name=f'encoder_conv_{i}'
            )(x)

            if self.use_batch_norm:
                x = layers.BatchNormalization()(x)

            x = layers.Activation("relu")(x)

            if self.use_dropout:
                x = layers.Dropout(0.25)(x)

        shape_before_flattening = K.int_shape(x)[1:]
        x = layers.Flatten()(x)

        embedding = layers.Dense(self.z_dim, name="Embedding")(x)

        # Decoder
        x = layers.Dense(np.prod(shape_before_flattening))(embedding)
        x = layers.Reshape(shape_before_flattening)(x)

        for i in range(self.n_layers_decoder):
            x = layers.Conv2DTranspose(
                filters=self.decoder_conv_t_filters[i],
                kernel_size=self.decoder_conv_t_kernel_size[i],
                strides=self.decoder_conv_t_strides[i],
                padding='same',
                name=f'decoder_conv_t_{i}'
            )(x)

            if i < self.n_layers_decoder - 1:
                if self.use_batch_norm:
                    x = layers.BatchNormalization()(x)
                x = layers.Activation('relu')(x)
                if self.use_dropout:
                    x = layers.Dropout(0.25)(x)
            else:
                x = layers.Activation('sigmoid')(x)

        autoencoder = Model(inputs=encoder_input, outputs=x, name="Face_Autoencoder")
        autoencoder.summary()

        return autoencoder

In [ ]:
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)

fae = Face_Autoencoder(
    input_dim=IMG_SHAPE,
    encoder_conv_filters=[32, 64, 64, 64],
    encoder_conv_kernel_size=[3, 3, 3, 3],
    encoder_conv_strides=[2, 2, 2, 2],
    decoder_conv_t_filters=[64, 64, 32, 3],
    decoder_conv_t_kernel_size=[3, 3, 3, 3],
    decoder_conv_t_strides=[2, 2, 2, 2],
    z_dim=128,
    use_batch_norm=False,
    use_dropout=False
)

face_ae = fae.build()

In [ ]:
fixed_imgs, _ = next(val_gen)
fixed_imgs = fixed_imgs[:5]  # keep first 5

In [ ]:
class ShowFixed(Callback):
    def __init__(self, fixed_images):
        super().__init__()
        self.fixed_images = fixed_images

    def on_epoch_end(self, epoch, logs=None):
        recon = face_ae.predict(self.fixed_images, verbose=0)

        n = len(self.fixed_images)
        plt.figure(figsize=(20, 6))
        for i in range(n):
            ax = plt.subplot(2, n, i + 1)
            plt.imshow(self.fixed_images[i])
            plt.axis("off")
            plt.title("Original")

            ax = plt.subplot(2, n, i + 1 + n)
            plt.imshow(recon[i])
            plt.axis("off")
            plt.title("Reconstructed")

        plt.show()

show_fixed = ShowFixed(fixed_imgs)

In [ ]:
face_ae.compile(
    loss='mse',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)
)

history = face_ae.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[show_fixed]
)

In [ ]:
# Storing our training and validation losses
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = [d for d in range(1, EPOCHS + 1)]

# Code for plotting train and val loss
plt.figure(figsize=(18, 10))
plt.plot(epochs, train_loss, '-r', label="Training")
plt.plot(epochs, val_loss, '--b', label="Validation")
plt.title("LOSS Curve for Face reconstruction")
plt.grid(True)
plt.xlabel('Epochs')
plt.ylabel("LOSS")
plt.legend()
plt.xticks(epochs)
plt.show()

In [ ]:
# Get a batch from the validation generator
imgs, _ = next(val_gen)

# Predict reconstructions for this batch
decoded_imgs = face_ae.predict(imgs)

n = 10
plt.figure(figsize=(20, 4))

for i in range(n):
    # pick a random index in the batch
    idx = np.random.randint(0, imgs.shape[0])

    # display original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(imgs[idx])
    plt.title("Original")
    plt.axis("off")

    # display reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[idx])
    plt.title("Reconstructed")
    plt.axis("off")

plt.show()

Face Recognition Using Autoencoder

In [ ]:
for i, layer in enumerate(face_ae.layers):
    print(i, layer.name)

In [ ]:
encoder = Model(face_ae.input, face_ae.layers[10].output, name="Encoder")
encoder.summary()

In [ ]:
# Pick 3 identities from LFW (example)
ids = ["Ahmad_Masood", "Aitor_Gonzalez", "Akbar_Hashemi_Rafsanjani"]

data_dir = "/kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"

In [ ]:
from glob import glob

id_paths = {}
for identity in ids:
    files = glob(os.path.join(data_dir, identity, "*.jpg"))
    id_paths[identity] = files
    print(identity, "num images:", len(files))

In [ ]:
import cv2
import numpy as np

def preprocess_img(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128, 128))
    img = img.astype("float32") / 255.0
    return img

In [ ]:
N = 10  # how many per identity to average

id_embs = {}  # per-identity list of embeddings

for identity, paths in id_paths.items():
    encs = []
    for p in paths[:N]:
        img = preprocess_img(p)
        emb = encoder.predict(np.array([img]))  # shape (1, z_dim)
        encs.append(emb[0])
    id_embs[identity] = np.stack(encs)  # shape (N, z_dim)

In [ ]:
id_means = {}
for identity, arr in id_embs.items():
    id_means[identity] = np.mean(arr, axis=0)

In [ ]:
rand_val = {}
for identity in ids:
    files = id_paths[identity]
    choice = np.random.choice(files)
    rand_val[identity] = choice

In [ ]:
from scipy.spatial.distance import euclidean

results = {}

for identity, img_path in rand_val.items():
    img = preprocess_img(img_path)
    emb = encoder.predict(np.array([img]))[0]

    # compute distance to each identity’s base
    dists = {k: euclidean(v, emb) for k, v in id_means.items()}
    results[identity] = dists

In [ ]:
import matplotlib.pyplot as plt

for identity, distdict in results.items():
    print(f"\nQuery image: {identity}")
    print(f"Distances =", distdict)

    # show query + base examples
    plt.figure(figsize=(12,4))

    # Query
    plt.subplot(1, 4, 1)
    plt.imshow(preprocess_img(rand_val[identity]))
    plt.title("Query " + identity)
    plt.axis("off")

    # Base images
    for i, (other_id, mean_emb) in enumerate(id_means.items()):
        plt.subplot(1, 4, i+2)
        # show any representative image
        plt.imshow(preprocess_img(id_paths[other_id][0]))
        plt.title(f"{other_id}\ndist={distdict[other_id]:.3f}")
        plt.axis("off")

    plt.show()